In [2]:
from pyspark.sql import SparkSession

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("PySpark-Practice") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create sample data
data = [("Alice", 34), ("Bob", 45), ("Charlie", 29)]
columns = ["Name", "Age"]

# 3. Create DataFrame
df = spark.createDataFrame(data, columns)

# 4. Show the DataFrame
df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 14:28:44 WARN Utils: Your hostname, codespaces-d0a7aa, resolves to a loopback address: 127.0.0.1; using 10.0.3.77 instead (on interface eth0)
26/10/28 14:28:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/python/3.14.2/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/28 14:28:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 34|
|    Bob| 45|
|Charlie| 29|
+-------+---+



#### Reading data

In [ ]:
# Read fact table

df_2020 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2020.csv", 
    header=True, 
    inferSchema=True
)

df_2021 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2021.csv", 
    header=True, 
    inferSchema=True
)

df_2022 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2022.csv", 
    header=True, 
    inferSchema=True
)

df = df_2020.unionByName(df_2021).unionByName(df_2022)

df.show()


+----------+----------+-----------+----------+-----------+------------+-------------+-------------+
| OrderDate| StockDate|OrderNumber|ProductKey|CustomerKey|TerritoryKey|OrderLineItem|OrderQuantity|
+----------+----------+-----------+----------+-----------+------------+-------------+-------------+
|2020-01-01|2019-09-21|    SO45080|       332|      14657|           1|            1|            1|
|2020-01-01|2019-12-05|    SO45079|       312|      29255|           4|            1|            1|
|2020-01-01|2019-10-29|    SO45082|       350|      11455|           9|            1|            1|
|2020-01-01|2019-11-16|    SO45081|       338|      26782|           6|            1|            1|
|2020-01-02|2019-12-15|    SO45083|       312|      14947|          10|            1|            1|
|2020-01-02|2019-10-12|    SO45084|       310|      29143|           4|            1|            1|
|2020-01-02|2019-12-18|    SO45086|       314|      18747|           9|            1|            1|


In [6]:
#read lookup table

df_cust = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Customer Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_cust.show()

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+---------------+--------------+---------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren| EducationLevel|    Occupation|HomeOwner|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+---------------+--------------+---------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|       90000|            2|      Bachelors|  Professional|        Y|
|      11001|   MR.|   EUGENE|   HUANG|1965-05-14|            S|     M|eugene10@adventur...|       60000|            3|      Bachelors|  Professional|        N|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|       60000|            3|      Bachelors|  Professional|        Y|
|      11003|   MS.|  CHRISTY|    

In [8]:
df_territory = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Territory Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_territory.show()

+-----------------+--------------+--------------+-------------+
|SalesTerritoryKey|        Region|       Country|    Continent|
+-----------------+--------------+--------------+-------------+
|                1|     Northwest| United States|North America|
|                2|     Northeast| United States|North America|
|                3|       Central| United States|North America|
|                4|     Southwest| United States|North America|
|                5|     Southeast| United States|North America|
|                6|        Canada|        Canada|North America|
|                7|        France|        France|       Europe|
|                8|       Germany|       Germany|       Europe|
|                9|     Australia|     Australia|      Pacific|
|               10|United Kingdom|United Kingdom|       Europe|
+-----------------+--------------+--------------+-------------+



In [10]:
df_product = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product.show()

+----------+---------------------+----------+--------------------+--------------------+--------------------+------------+-----------+------------+-----------+------------+
|ProductKey|ProductSubcategoryKey|ProductSKU|         ProductName|           ModelName|  ProductDescription|ProductColor|ProductSize|ProductStyle|ProductCost|ProductPrice|
+----------+---------------------+----------+--------------------+--------------------+--------------------+------------+-----------+------------+-----------+------------+
|       214|                   31| HL-U509-R|Sport-100 Helmet,...|           Sport-100|Universal fit, we...|         Red|          0|           0|    13.0863|       34.99|
|       215|                   31|   HL-U509|Sport-100 Helmet,...|           Sport-100|Universal fit, we...|       Black|          0|           0|    12.0278|     33.6442|
|       218|                   23| SO-B909-M|Mountain Bike Soc...| Mountain Bike Socks|Combination of na...|       White|          M|       

In [11]:
df_product_subcat = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Subcategories Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product_subcat.show()

+---------------------+---------------+------------------+
|ProductSubcategoryKey|SubcategoryName|ProductCategoryKey|
+---------------------+---------------+------------------+
|                    1| Mountain Bikes|                 1|
|                    2|     Road Bikes|                 1|
|                    3|  Touring Bikes|                 1|
|                    4|     Handlebars|                 2|
|                    5|Bottom Brackets|                 2|
|                    6|         Brakes|                 2|
|                    7|         Chains|                 2|
|                    8|      Cranksets|                 2|
|                    9|    Derailleurs|                 2|
|                   10|          Forks|                 2|
|                   11|       Headsets|                 2|
|                   12|Mountain Frames|                 2|
|                   13|         Pedals|                 2|
|                   14|    Road Frames|                 

In [12]:
df_product_cat = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Categories Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product_cat.show()

+------------------+------------+
|ProductCategoryKey|CategoryName|
+------------------+------------+
|                 1|       Bikes|
|                 2|  Components|
|                 3|    Clothing|
|                 4| Accessories|
+------------------+------------+



## Practice Questions

Goal: From df_cust, create a new column called FullName by concatenating FirstName and LastName with a space in between. Then, filter for customers who own a home (HomeOwner == 'Y') and earn an AnnualIncome strictly greater than 80,000.

In [19]:
from pyspark.sql import functions as F

df_cust_copy = df_cust.withColumn("FullName", F.concat_ws(" ", F.col("FirstName"), F.col("LastName")))
df_cust_copy = df_cust_copy.filter(F.col("HomeOwner")=="Y")

df_cust_copy.show()

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+-------------------+--------------+---------+-----------------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|     EducationLevel|    Occupation|HomeOwner|         FullName|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+-------------------+--------------+---------+-----------------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|       90000|            2|          Bachelors|  Professional|        Y|         JON YANG|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|       60000|            3|          Bachelors|  Professional|        Y|     RUBEN TORRES|
|      11004|  MRS.|ELIZABETH| JOHNSON|1968-08-08|            S|     F|elizabeth5@adv

Goal: In df_product, create a column ProfitMargin = ProductPrice - ProductCost. Display the top 10 products with the highest profit margin, sorted descending. Exclude products where ProductPrice or ProductCost is 0 or null.

In [22]:
df_product_copy = df_product.withColumn("Profit Margin", F.col("ProductPrice") - F.col("ProductCost"))
df_product_copy = df_product_copy.filter(
    (F.col("ProductPrice") != 0) 
    & (F.col("ProductCost") != 0 )
    & (F.col("ProductPrice").isNotNull())
    & (F.col("ProductCost").isNotNull() ))
df_product_copy.orderBy(F.col("Profit Margin").desc()).limit(10).show()

+----------+---------------------+----------+--------------------+------------+--------------------+------------+-----------+------------+-----------+------------+------------------+
|ProductKey|ProductSubcategoryKey|ProductSKU|         ProductName|   ModelName|  ProductDescription|ProductColor|ProductSize|ProductStyle|ProductCost|ProductPrice|     Profit Margin|
+----------+---------------------+----------+--------------------+------------+--------------------+------------+-----------+------------+-----------+------------+------------------+
|       344|                    1|BK-M82S-38|Mountain-100 Silv...|Mountain-100|Top-of-the-line c...|      Silver|         38|           U|  1912.1544|     3399.99|1487.8355999999999|
|       345|                    1|BK-M82S-42|Mountain-100 Silv...|Mountain-100|Top-of-the-line c...|      Silver|         42|           U|  1912.1544|     3399.99|1487.8355999999999|
|       346|                    1|BK-M82S-44|Mountain-100 Silv...|Mountain-100|Top-of

In df, extract the Year and Month from OrderDate. Count how many total orders were placed in each month of the year 2020. Sort chronologically by month.

In [26]:
df_copy = df.withColumn("Month",F.month(F.col("OrderDate"))) \
            .withColumn("Year",F.year(F.col("OrderDate"))) \
            .filter(F.col("Year") == 2020) \
            .groupBy("Month").agg(F.count("OrderNumber")) \
            .orderBy(F.col("Month").asc())
df_copy.show()

+-----+------------------+
|Month|count(OrderNumber)|
+-----+------------------+
|    1|               184|
|    2|               165|
|    3|               198|
|    4|               204|
|    5|               206|
|    6|               212|
|    7|               247|
|    8|               278|
|    9|               196|
|   10|               223|
|   11|               191|
|   12|               326|
+-----+------------------+



Using df, find the count of distinct CustomerKey values that placed orders in each TerritoryKey

In [28]:
df_copy = df.groupBy("TerritoryKey").agg(F.countDistinct("CustomerKey"))
df_copy.show()

+------------+---------------------------+
|TerritoryKey|count(DISTINCT CustomerKey)|
+------------+---------------------------+
|           1|                       3075|
|           6|                       1499|
|           3|                          8|
|           5|                         10|
|           9|                       3480|
|           4|                       4134|
|           8|                       1675|
|           7|                       1705|
|          10|                       1822|
|           2|                          8|
+------------+---------------------------+



Goal: Join df and df_product on ProductKey. Calculate total sales revenue per ProductName:

Revenue = OrderQuantity × ProductPrice

Display the top 5 highest-grossing products.

In [40]:
df_merged = df.join(df_product, on = "ProductKey")\
            .select(*df.columns, 'ProductName','ProductPrice')\
            .withColumn("Revenue",F.col("OrderQuantity") * F.col('ProductPrice'))\
            .groupBy("ProductName").agg(F.sum("Revenue").alias("Revenue_Per_Product"))\
            .orderBy(F.col("Revenue_Per_Product").desc())\
            .limit(5)

df_merged.show()

+--------------------+-------------------+
|         ProductName|Revenue_Per_Product|
+--------------------+-------------------+
|Mountain-200 Blac...| 1241753.5092000025|
|Mountain-200 Blac...| 1233557.1164000025|
|Mountain-200 Silv...| 1213851.8856000048|
|Mountain-200 Silv...| 1182780.5916000048|
|Mountain-200 Blac...| 1165936.8758000024|
+--------------------+-------------------+



Goal: Join all 4 tables together:
df → df_product → df_product_subcat → df_product_cat.

Calculate total revenue and total quantity sold for each CategoryName (e.g., Bikes, Clothing, Components, Accessories).

Key functions: Multi-table chaining joins.

In [46]:
df_merged = df.join(df_product,on = "ProductKey")\
            .join(df_product_subcat, on = "ProductSubcategoryKey")\
            .join(df_product_cat, on = "ProductCategoryKey")\
            .withColumn("Revenue",F.col("OrderQuantity") * F.col('ProductPrice'))\
            .groupBy("CategoryName").agg(F.sum("Revenue").alias("total_revenue"),F.sum("OrderQuantity").alias("total_orders"))

df_merged.show()

+------------+--------------------+------------+
|CategoryName|       total_revenue|total_orders|
+------------+--------------------+------------+
|       Bikes|2.3642495095200203E7|       13929|
|    Clothing|   365418.6170999935|       12436|
| Accessories|   906673.1069998639|       57809|
+------------+--------------------+------------+



Find all customers in df_cust who have never placed any order in df. Display their CustomerKey, FirstName, LastName, and EmailAddress.

In [ ]:
df_copy = df.withColumn(
    "CustomerKey",
    F.col("CustomerKey").cast("string")
)

df_merged = df_cust.join(df_copy, on="CustomerKey")\
            .filter(F.col("OrderNumber").isNull())

df_merged.show()

+-----------+------+---------+--------+---------+-------------+------+------------+------------+-------------+--------------+----------+---------+---------+---------+-----------+----------+------------+-------------+-------------+
|CustomerKey|Prefix|FirstName|LastName|BirthDate|MaritalStatus|Gender|EmailAddress|AnnualIncome|TotalChildren|EducationLevel|Occupation|HomeOwner|OrderDate|StockDate|OrderNumber|ProductKey|TerritoryKey|OrderLineItem|OrderQuantity|
+-----------+------+---------+--------+---------+-------------+------+------------+------------+-------------+--------------+----------+---------+---------+---------+-----------+----------+------------+-------------+-------------+
+-----------+------+---------+--------+---------+-------------+------+------------+------------+-------------+--------------+----------+---------+---------+---------+-----------+----------+------------+-------------+-------------+



Using df_cust, group by Occupation and calculate:

Average AnnualIncome.

Total number of customers.

Percentage of customers who are married (MaritalStatus == 'M').

In [62]:
df_cust_copy = df_cust.groupBy("Occupation") \
    .agg(
        F.avg("AnnualIncome").alias("Average Annual Income"),
        F.count("*").alias("Customer Count"),
        F.count_if(F.col("MaritalStatus") == "M").alias("Married Count")
    ) \
    .withColumn("Married Pct",F.col("Married Count")/F.col("Customer Count"))

df_cust_copy.show()

+--------------+---------------------+--------------+-------------+-------------------+
|    Occupation|Average Annual Income|Customer Count|Married Count|        Married Pct|
+--------------+---------------------+--------------+-------------+-------------------+
|    Management|     92218.5320491531|          3011|         1888| 0.6270342079043507|
|          NULL|                 NULL|             6|            0|                0.0|
|  Professional|    74166.66666666667|          5424|         3062|   0.56452802359882|
|      Clerical|     30696.0475690801|          2859|         1407|0.49213011542497376|
|        Manual|   16472.588185295368|          2353|          791| 0.3361665958351041|
|Skilled Manual|    51732.94823372584|          4501|         2669| 0.5929793379249055|
+--------------+---------------------+--------------+-------------+-------------------+

